# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimlazrek1/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup (Local)

In [1]:
import os
from pathlib import Path

import duckdb

ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    import getpass
    print("Tip: pip install python-dotenv")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("Connected.")

Connected.


## 1. My rule and its reason codes

**Lane 4 — CTR / Engagement Opportunity Scoring**

Same slice as ML-04: warehouse **March 2026** (`month=2026-03`), one row per page, `imp_mar >= 100`, real position only.

### My rule

> Rank pages **higher** when they have **enough March impressions** and their **`ctr_mar` is below the median for their `position_tier`**. A reviewer opens the top of that list first.

### Two signal checks

Before scoring pages, we check two ideas the rule depends on, at least one must be a signal behind a real FlyRank flag from the session :  
Example: `low_ctr_visible_page` = impressions ≥ 500, position 1–20, CTR < 0.5 (from the docs).


| Signal | What we check | FlyRank idea it mirrors |
|---|---|---|
| **1. CTR vs position** | Does CTR change by `position_tier` | Similar to their `low_ctr_visible_page` flag — “visible page, weak CTR.” |
| **2. Volume** | Are low-impression pages noisier? | Similar to their impression floor (With few impressions, CTR jumps around)|

### Reason codes 
*Labels that explain why a page is on the ranked list*

| Reason code | When (first match wins) |
|---|---|
| `high_visibility_ctr_gap` | weak ctr + `imp_mar >= 500` -- top priority bc it hits both problems |
| `ctr_below_tier_median` | weak ctr |
| `general_ctr_monitor` | everything else on the list |

### Signal verdicts
*Giving each a one-word verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE*

**Signal 1 — CTR vs position:** `CONFIRMED`  
Weighted CTR falls from 0.41% on `top_3` to 0.04% on `deep` (n = 8,295 and 3,949 pages), so CTR must be compared within `position_tier`, not site-wide.

**Signal 2 — Volume:** `CONFIRMED`  
Low-impression bands are noisier: 74% zero-click pages in `100-299` (n = 26,775) vs 3% in `3000+` (n = 22,157), so a minimum impression floor is needed before we trust CTR.    
   
*n = how many pages are in that bucket.*


In [2]:
import pandas as pd

TIER_ORDER = ["top_3", "page_1", "striking", "page_3_5", "deep"]
IMP_BANDS = [100, 300, 500, 1000, 3000, float("inf")]
IMP_LABELS = ["100-299", "300-499", "500-999", "1000-2999", "3000+"]

features = con.sql(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_mar,
            SUM(gsc_clicks) AS clk_mar,
            AVG(NULLIF(gsc_avg_position, 0)) AS pos_avg_mar
        FROM {FACT_MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),
    scored AS (
        SELECT
            *,
            CASE WHEN imp_mar > 0 THEN 100.0 * clk_mar / imp_mar END AS ctr_mar,
            CASE
                WHEN pos_avg_mar <= 3 THEN 'top_3'
                WHEN pos_avg_mar <= 10 THEN 'page_1'
                WHEN pos_avg_mar <= 20 THEN 'striking'
                WHEN pos_avg_mar <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier
        FROM daily
        WHERE pos_avg_mar > 0
    ),
    labeled AS (
        SELECT
            s.*,
            MEDIAN(ctr_mar) OVER (PARTITION BY position_tier) AS tier_median_ctr,
            CASE
                WHEN ctr_mar < MEDIAN(ctr_mar) OVER (PARTITION BY position_tier)
                     AND imp_mar >= 500
                THEN 1
                ELSE 0
            END AS is_ctr_underperformer
        FROM scored s
    )
    SELECT * FROM labeled
""").df()

print(f"Lane slice n = {len(features):,}")

print("\nSIGNAL 1 — CTR vs position (by position_tier)")
sig1 = (
    features.groupby("position_tier", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        sum_imp=("imp_mar", "sum"),
        sum_clk=("clk_mar", "sum"),
    )
    .assign(weighted_ctr_mar=lambda d: 100.0 * d["sum_clk"] / d["sum_imp"])
    .drop(columns=["sum_imp", "sum_clk"])
    .reindex(TIER_ORDER)
    .reset_index()
)
print(sig1.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSIGNAL 2 — Volume (by impression band)")
features["imp_band"] = pd.cut(
    features["imp_mar"], bins=IMP_BANDS, labels=IMP_LABELS, right=False
)
sig2 = (
    features.groupby("imp_band", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        share_zero_clicks=("ctr_mar", lambda s: (s == 0).mean()),
    )
    .reset_index()
)
print(sig2.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

Lane slice n = 101,441

SIGNAL 1 — CTR vs position (by position_tier)
position_tier     n  weighted_ctr_mar
        top_3  8295            0.4067
       page_1 46531            0.3232
     striking 21738            0.3052
     page_3_5 20928            0.1361
         deep  3949            0.0356

SIGNAL 2 — Volume (by impression band)
 imp_band     n  share_zero_clicks
  100-299 26775             0.7393
  300-499 12742             0.5642
  500-999 16866             0.3801
1000-2999 22901             0.1624
    3000+ 22157             0.0301


## 2. Build the ranked queue (writes the CSV)

**Score:** `baseline_score = ctr_gap` (tier median CTR − page CTR).  

*Writes `work/outputs/baseline_action_score.csv`.*

In [3]:
OUT = ROOT / "work" / "outputs" / "baseline_action_score.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

queue = features.copy()
queue["ctr_gap"] = queue["tier_median_ctr"] - queue["ctr_mar"]
queue["baseline_score"] = queue["ctr_gap"]

def reason_code(row) -> str:
    if row["is_ctr_underperformer"] == 1 and row["imp_mar"] >= 500:
        return "high_visibility_ctr_gap"
    if row["is_ctr_underperformer"] == 1:
        return "ctr_below_tier_median"
    return "general_ctr_monitor"

def action_label(code: str) -> str:
    if code == "high_visibility_ctr_gap":
        return "refresh_and_review_ctr"
    if code == "ctr_below_tier_median":
        return "review_ctr"
    return "monitor"

queue["reason_code"] = queue.apply(reason_code, axis=1)
queue["action"] = queue["reason_code"].map(action_label)
queue["baseline_rank"] = queue["baseline_score"].rank(method="first", ascending=False).astype(int)

export_cols = [
    "baseline_rank", "content_hash_id", "client_hash_id",
    "imp_mar", "ctr_mar", "pos_avg_mar", "position_tier",
    "baseline_score", "reason_code", "action",
]
queue.sort_values("baseline_rank").to_csv(OUT, index=False, columns=export_cols)
print(f"Wrote {len(queue):,} rows → {OUT}")


Wrote 101,441 rows → c:\Users\rimla\Desktop\work_folder\flyrank-internship\work\outputs\baseline_action_score.csv


### Baseline metrics

**Label** = `is_ctr_underperformer` (CTR below tier median and `imp_mar >= 500`)  
**Score** = `ctr_gap`   
**Metric** = Precision@K — of the top K pages our rule ranks, how many are underperformers?

In [4]:
import json
import numpy as np

LABEL = "is_ctr_underperformer"

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

labels = queue[LABEL].to_numpy()
scores = queue["baseline_score"].to_numpy()
base_rate = float(labels.mean())

metrics = {
    "method": "baseline_rule",
    "label": LABEL,
    "slice": "month=2026-03",
    "n": int(len(queue)),
    "base_rate": base_rate,
    "precision_at_10": precision_at_k(scores, labels, 10),
    "precision_at_20": precision_at_k(scores, labels, 20),
    "precision_at_50": precision_at_k(scores, labels, 50),
}

METRICS_OUT = ROOT / "work" / "outputs" / "baseline_metrics.json"
METRICS_OUT.write_text(json.dumps(metrics, indent=2))

print(f"Base rate (share {LABEL}=1): {base_rate:.3f}")
for k in (10, 20, 50):
    print(f"Precision@{k}: {metrics[f'precision_at_{k}']:.3f}")
print(f"Saved → {METRICS_OUT}")

Base rate (share is_ctr_underperformer=1): 0.221
Precision@10: 0.200
Precision@20: 0.500
Precision@50: 0.480
Saved → c:\Users\rimla\Desktop\work_folder\flyrank-internship\work\outputs\baseline_metrics.json


## 3. Top-10 review

For each of the top 10: **action**, **why it's here**, and **what would make it wrong**.

With `baseline_score = ctr_gap`, many `top_3` zero-click pages **tie** on the same max gap (~0.25). Rank order among ties is arbitrary — impressions do not break ties anymore.

| Rank | action | reason_code | why it's here | what would make it wrong |
|---:|---|---|---|---|
| 1 | review_ctr | ctr_below_tier_median | `top_3`, **0% CTR**, ~270 March impressions — max `ctr_gap` vs `top_3` median (~0.25%) | Zero clicks on ~270 imp may be noise; rank 4–5/8–9 have same gap with more traffic |
| 2 | review_ctr | ctr_below_tier_median | `top_3`, **0% CTR**, ~419 impressions — same max gap, below tier median | Same tie issue; low volume makes 0% CTR less trustworthy |
| 3 | review_ctr | ctr_below_tier_median | `top_3`, **0% CTR**, ~387 impressions — tied max `ctr_gap` | Tracking gap or SERP answers on-page, not snippet quality |
| 4 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, ~714 impressions — max gap **and** imp ≥ 500 | Clicks exist in daily data but rolled to zero; URL/tracking issue |
| 5 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, ~1.5k impressions — strongest visibility among tied max-gap rows | Brand/nav query where clicks are not expected |
| 6 | review_ctr | ctr_below_tier_median | `top_3`, **0% CTR**, ~155 impressions — tied score but thin traffic | Signal 2: 155 imp is noisy; not a clear fix opportunity |
| 7 | review_ctr | ctr_below_tier_median | `top_3`, **0% CTR**, ~123 impressions — tied at top despite floor-level volume | Almost certainly noise; should not outrank rank 5 or 8 |
| 8 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, ~1.7k impressions — high visibility + max gap | Featured snippet or zero-click SERP layout |
| 9 | refresh_and_review_ctr | high_visibility_ctr_gap | `top_3`, **0% CTR**, ~1.6k impressions — same pattern as rank 8 | Intent mismatch; page ranks but users don't click |
| 10 | review_ctr | ctr_below_tier_median | `top_3`, **0% CTR**, ~285 impressions — tied max gap, moderate-low volume | Seasonal March dip, not a page-specific CTR problem |

In [5]:
top10_cols = [
    "baseline_rank", "content_hash_id", "imp_mar", "ctr_mar",
    "position_tier", "reason_code", "action",
]
print(queue.sort_values("baseline_rank").head(10)[top10_cols].to_string(index=False))

 baseline_rank          content_hash_id  imp_mar  ctr_mar position_tier             reason_code                 action
             1 content_bba10cacad00a26c    423.0      0.0         top_3     general_ctr_monitor                monitor
             2 content_886d1ba05d3dcfbb    211.0      0.0         top_3     general_ctr_monitor                monitor
             3 content_101ca4884860be65   1355.0      0.0         top_3 high_visibility_ctr_gap refresh_and_review_ctr
             4 content_cc4682e245b75d64    666.0      0.0         top_3 high_visibility_ctr_gap refresh_and_review_ctr
             5 content_34b2d480f97810ba    705.0      0.0         top_3 high_visibility_ctr_gap refresh_and_review_ctr
             6 content_8e3eb32250ca5f79    165.0      0.0         top_3     general_ctr_monitor                monitor
             7 content_fdc75e70679a0ba4    615.0      0.0         top_3 high_visibility_ctr_gap refresh_and_review_ctr
             8 content_772d7e0805c3ab52    420.0

## 4. Weak picks + leakage check

**Weak picks** *(1–2 ranks from the top 10 that look shaky and why)*:

1. **Rank 7** — ~123 impressions and 0% CTR on `top_3`. Tied on max `ctr_gap`, but Signal 2 says low-impression zero-click rows are noisy — rank 5 or 8 (1.5k+ imp) is a stronger pick at the same score.
2. **Rank 6** — ~155 impressions, same tied max gap. Thin traffic makes 0% CTR hard to trust; a human would open rank 4–5 or 8–9 first.

**Leakage check (plain words):**

- No product flags (`needs_ctr_fix`, `health_score`, …) — not in the data ✓
- No future-window columns — March 2026 only ✓
- No `trend_direction` / `trend_pct` in the score ✓
- `ctr_gap` is **part of the baseline rule**, not a model feature — OK for this hand rule; dropped for ML-08 models (see ML-04 trap)

In [6]:
FORBIDDEN = {"trend_direction", "trend_pct", "health_score", "needs_ctr_fix", "priority_score"}
print("Forbidden cols present:", sorted(FORBIDDEN & set(queue.columns)) or "none (good)")

Forbidden cols present: none (good)


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.